# Feature engineering

Builds the full gold feature table from silver transactions and inspects what each feature group adds: transaction, temporal, entity, and graph features.

**Prerequisites:** `make sample-data` and `make ingest`.

In [ ]:
from transaction_risk.spark.session import create_spark_session_from_yaml
spark = create_spark_session_from_yaml('../conf/spark.local.yaml')


In [ ]:
from transaction_risk.features.pipeline import build_feature_table
from transaction_risk.spark.io import read_table

transactions = read_table(spark, '../data/silver/transactions')
features = build_feature_table(transactions, feature_config_path='../conf/features.yaml')

added_columns = sorted(set(features.columns) - set(transactions.columns))
print(f'{len(added_columns)} feature columns added:')
for column in added_columns:
    print(' -', column)

In [ ]:
# Temporal features only look backwards in time: history columns are 0/-1 for an account's first transaction
features.select(
    'nameOrig',
    'step',
    'amount',
    'steps_since_previous_origin_tx',
    'origin_tx_count_before',
    'origin_amount_mean_before',
    'origin_amount_zscore_before',
).orderBy('nameOrig', 'step').show(10)

In [ ]:
# The feature registry documents ownership, sources, and leakage risk for every feature
from transaction_risk.features.metadata import get_feature_registry

registry = get_feature_registry()
print(f'{len(registry)} registered features. Example entry:')
print(registry[0])
spark.stop()